# Run inference for Qwen3TTS

This notebook demonstrates how to:
1. Initialize the Qwen3TTS model with dummy weights
2. Run inference with random embeddings to verify the model works


In [ ]:
import os
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["VLLM_DISABLE_REQUEST_ID_RANDOMIZATION"] = "1"
#os.environ["VLLM_DISABLE_PAD_FOR_CUDAGRAPH"] = "1"


In [ ]:
import torch
from pathlib import Path
from vllm import SamplingParams
from vllm.engine.arg_utils import AsyncEngineArgs
from vllm.v1.engine.async_llm import AsyncLLM

In [ ]:
# Load vLLM engine with dummy model
import json

type_str = "bfloat16"
torch_type = getattr(torch, type_str)

max_len = 256
config_path = Path("dummy_qwen3_tts_model")
engine_args = AsyncEngineArgs(
    model=str(config_path.absolute()),
    dtype=type_str,
    max_model_len=max_len,
    max_num_batched_tokens=max_len,
    gpu_memory_utilization=0.6,
    skip_tokenizer_init=True,
    enable_prefix_caching=False,
    trust_remote_code=True,
    #enforce_eager=True,
    #compilation_config={"cudagraph_mode": "PIECEWISE"},
    input_coalesce_timeout_ms=5,
    shm_decode=True,
    attention_backend="TRITON_ATTN",
)

print("Initializing engine...")
engine = AsyncLLM.from_engine_args(engine_args)

with open(config_path / "config.json") as f:
    _cfg = json.load(f)
sampling_params = SamplingParams(
    max_tokens=max_len,
    #temperature=_cfg.get("temperature", 0.9),
    temperature=0.9,
    top_k=_cfg.get("top_k", 50),
    top_p=_cfg.get("top_p", 1.0),
    repetition_penalty=_cfg.get("repetition_penalty", 1.0),
)

print("Engine initialized successfully")


In [ ]:
# prepare prefill tokens
from transformers import AutoTokenizer
from vllm.model_executor.models.qwen3_tts import Qwen3TTSTalkerForConditionalGeneration


with open(config_path / "config.json") as f:
  config = json.load(f)
tokenizer = AutoTokenizer.from_pretrained(str(config_path.absolute()))
text_ids, group0_ids = Qwen3TTSTalkerForConditionalGeneration.build_prefill_tokens(
  tokenizer=tokenizer,
  text="Another example, to see that everything is actually fine before moving this to the server as well.",
  speaker="aiden",
  language="english",
  config=config,
)
print(f"Prefill text_ids {text_ids.shape}, group0_ids {group0_ids.shape}")

In [ ]:
request_id = "test_request_1"

tc = config["talker_config"]
codec_eos_token_id = tc["codec_eos_token_id"]
tts_pad_token_id = config["tts_pad_token_id"]
hidden_size = tc["hidden_size"]

prompt_len = text_ids.shape[0]

inputs = {
    "prompt_token_ids": group0_ids.tolist(),
    "custom_inputs": {
        "text_ids": text_ids,
        "prev_hidden": torch.zeros(prompt_len, hidden_size, dtype=torch.bfloat16),
    },
}

print(f"Starting generation with request_id: {request_id}")
print(f"Prompt length: {prompt_len}")

queue = await engine.add_request(request_id, inputs, sampling_params)
prefill_output = await queue.get()
custom_out = prefill_output.outputs[0].custom_outputs

prev_hidden = custom_out["hidden"][-1:].clone()  # [1, hidden_size]
prev_group0 = prefill_output.outputs[0].token_ids[-1]


generated_codecs = []

decode_text_id = torch.tensor([tts_pad_token_id], dtype=torch.long)

for step in range(max_len - 1):
    outputs = engine.decode_step_shm(
        request_id,
        custom_inputs={
            "text_ids": decode_text_id,
            "prev_hidden": prev_hidden,
        },
    )

    codes_1_15 = outputs["codes"][-1:].clone()  # [1, 15]
    prev_hidden = outputs["hidden"][-1:].clone()  # [1, hidden_size]

    # Assemble complete codec frame: [group0, codes_1_15]
    frame = torch.cat([
        torch.tensor([[prev_group0]], dtype=torch.long),
        codes_1_15,
    ], dim=-1)  # [1, 16]
    generated_codecs.append(frame)

    sampled_token = outputs.get("sampled_token_ids")
    #g0_tok = int(sampled_token[-1]) if sampled_token is not None else prev_group0
    g0_tok = int(sampled_token[-1])

    if g0_tok == codec_eos_token_id:
        print(f"EOS token detected at step {step + 1}, stopping generation.")
        await engine.abort(request_id)
        break

    prev_group0 = g0_tok

In [ ]:
if generated_codecs:
    arr = torch.cat(generated_codecs, dim=0)[:100]

    import matplotlib.pyplot as plt
    plt.imshow(arr.cpu().numpy().T, aspect='auto')
    plt.colorbar()
    plt.show()
else:
    print("No codec frames generated.")

In [ ]:
print(arr.shape)
print(torch.min(arr), torch.max(arr))
torch.save(arr, "/home/vklimkov/workspace/qwen3_tts/Qwen3-TTS/vllm_pred_tokens.pt")